# Εργαστήριο 2: ποιοι φοιτητές θα λάβουν τι;
**Προαιρετικό τετράδιο, για μετά το μάθημα.** Έκδοση διδάσκοντος, με σχολιασμό.

Δεδομένα: Realinho, Machado, Baptista & Martins (2022), *Data*, 7(11), 146· UCI Machine Learning Repository, σύνολο δεδομένων 697· άδεια CC BY 4.0. Πραγματικά, ανωνυμοποιημένα δεδομένα 4.424 φοιτητών.

## 1. Τα δεδομένα
Το αρχείο διαβάζεται απευθείας από το UCI. Η έκβαση έχει τρεις τιμές· εδώ προβλέπουμε το «εγκατέλειψε» έναντι όλων των άλλων.

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

URL = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
src = os.environ.get("LAB2_DATA", URL)            # στο Colab διαβάζεται απευθείας από το UCI
D = pd.read_csv(src, sep=";", compression="zip" if src.endswith(".zip") else None)
D.columns = [c.strip() for c in D.columns]
D["y"] = (D["Target"] == "Dropout").astype(int)     # 1 = εγκατέλειψε, 0 = όλοι οι άλλοι
print(D.shape, D["Target"].value_counts().to_dict())

## 2. Το ίδιο μοντέλο σε τρεις στιγμές (Τεκμήριο 1)

In [ ]:
CATS = ["Marital status", "Application mode", "Course", "Previous qualification", "Nacionality",
        "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]
s1 = [c for c in D.columns if "1st sem" in c]; s2 = [c for c in D.columns if "2nd sem" in c]
enrol = [c for c in D.columns if c not in s1 + s2 + ["Target", "y"]]
MOMENTS = {"κατά την εγγραφή": enrol, "τέλος 1ου εξαμήνου": enrol + s1, "τέλος 2ου εξαμήνου": enrol + s1 + s2}
tr, te = train_test_split(D.index, test_size=0.2, random_state=697, stratify=D["y"])   # 80% / 20%, όπως συνιστούν οι δημιουργοί

def fit(cols):
    X = pd.get_dummies(D[cols].astype({c: "category" for c in cols if c in CATS}), drop_first=True).astype(float)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.3)).fit(X.loc[tr], D["y"].loc[tr])
    return m, X, pd.Series(m.predict_proba(X.loc[te])[:, 1], index=te)

rows = []
for name, cols in MOMENTS.items():
    m, X, p = fit(cols)
    rows.append({"στιγμή": name, "μεταβλητές": len(cols), "AUC": round(roc_auc_score(D["y"].loc[te], p), 3), "ορθή κατάταξη": round(((p >= 0.5) == D["y"].loc[te]).mean(), 3)})
pd.DataFrame(rows)

> **Σχολιασμός.** Αναμένονται AUC 0.80, 0.89 και 0.91. Το κέρδος ενός εξαμήνου αναμονής είναι μεγάλο· το κέρδος του δεύτερου εξαμήνου μικρό. Ο σπόρος 697 και το C = 0,3 δίνουν ακριβώς τους αριθμούς του πακέτου.

## 3. Το κατώφλι (Τεκμήριο 2)

In [ ]:
model, X1, risk = fit(MOMENTS["τέλος 1ου εξαμήνου"])
T = D.loc[te].assign(risk=risk)                      # οι 885 φοιτητές ελέγχου

def perf(G, flag):
    tp = int((flag & (G.y == 1)).sum()); fp = int((flag & (G.y == 0)).sum()); fn = int((~flag & (G.y == 1)).sum()); tn = len(G) - tp - fp - fn
    return {"φοιτητές": len(G), "εγκατέλειψαν": round(G.y.mean(), 2), "επισημαίνονται": int(flag.sum()), "ορθά": tp, "χωρίς λόγο": fp, "διαφεύγουν": fn,
            "ευστοχία": round(tp / max(tp + fp, 1), 2), "ανάκληση": round(tp / max(tp + fn, 1), 2),
            "ψευδώς θετικοί %": round(100 * fp / max(fp + tn, 1)), "ψευδώς αρνητικοί %": round(100 * fn / max(fn + tp, 1))}

def sweep(thresholds=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7)):
    return pd.DataFrame({t: perf(T, T.risk >= t) for t in thresholds}).T

sweep()

> **Σχολιασμός.** 885 φοιτητές ελέγχου, 284 εγκατέλειψαν. Στο 0,4: 270 επισημάνσεις, 59 χωρίς λόγο, 73 διαφεύγουν. Ζητήστε από τους φοιτητές να δικαιολογήσουν το κατώφλι με την ενέργεια, όχι με την «ισορροπία» ευστοχίας και ανάκλησης.

## 4. Η δυναμικότητα: ποιοι βρίσκονται στην κορυφή της λίστας (Τεκμήριο 3)

In [ ]:
a1 = "Curricular units 1st sem (approved)"
rank = T.risk.rank(ascending=False, method="first")
for k in (50, 100):
    top = T[rank <= k]
    print(f"οι {k} πρώτοι: εγκατέλειψαν {int(top.y.sum())}, χωρίς κανένα επιτυχές μάθημα {int((top[a1] == 0).sum())}")
nz = T[T[a1] > 0]
print("με ένα τουλάχιστον επιτυχές μάθημα:", perf(nz, nz.risk >= 0.4))

> **Σχολιασμός.** Από τους 100 πρώτους εγκατέλειψαν οι 99, και οι 71 δεν είχαν περάσει κανένα μάθημα. Η κορυφή της λίστας είναι εκεί όπου το μοντέλο προσθέτει τα λιγότερα.

## 5. Οι υποομάδες (Τεκμήριο 4)
`Gender`: 1 = άνδρας, 0 = γυναίκα. Στις υπόλοιπες μεταβλητές ναι/όχι: 1 = ναι.

In [ ]:
def by_group(col, threshold=0.4):
    return pd.DataFrame({val: perf(G, G.risk >= threshold) for val, G in T.groupby(col)}).T

T["ηλικιακή ομάδα"] = pd.cut(T["Age at enrollment"], [0, 20, 25, 99], labels=["έως 20", "21 έως 25", "26 και άνω"]).astype(str)
for col in ["Gender", "Scholarship holder", "Tuition fees up to date", "ηλικιακή ομάδα"]:     # Gender: 1 = άνδρας, 0 = γυναίκα
    print("\n", col); print(by_group(col)[["φοιτητές", "εγκατέλειψαν", "επισημαίνονται", "ψευδώς θετικοί %", "ψευδώς αρνητικοί %"]])

> **Σχολιασμός.** Τα βασικά ποσοστά εγκατάλειψης διαφέρουν πολύ μεταξύ των ομάδων. Η ανισότητα των σφαλμάτων δεν οφείλεται στη μέτρηση, όπως στο Εργαστήριο 1, και δεν διορθώνεται: επιλέγεται. Ομάδες κάτω των 50 ατόμων δεν κρίνονται.

## 6. Τι βαραίνει στο μοντέλο (Τεκμήριο 5)
Τυποποιημένοι συντελεστές: συγκρίνονται μεταξύ τους, δεν ερμηνεύονται αιτιακά.

In [ ]:
w = pd.Series(model[-1].coef_[0], index=X1.columns)
w.reindex(w.abs().sort_values(ascending=False).index).head(10).round(2)

## 7. Δοκιμάστε

In [ ]:
# Δοκιμάστε: αλλάξτε το κατώφλι, ή ελέγξτε άλλη υποομάδα (π.χ. "Debtor", "Displaced", "Daytime/evening attendance", "Marital status")
by_group("Debtor", threshold=0.3)

## 8. Άσκηση: ελαχιστοποίηση δεδομένων
Αφαιρέστε τα κοινωνικοοικονομικά στοιχεία και εκπαιδεύστε ξανά. Πριν τρέξετε το κελί, γράψτε τι περιμένετε.

In [ ]:
# Άσκηση: το ίδιο μοντέλο ΧΩΡΙΣ τα κοινωνικοοικονομικά στοιχεία. Τι αλλάζει στην ακρίβεια; Τι αλλάζει για όσους καθυστερούν τα δίδακτρα;
SOCIO = ["Debtor", "Tuition fees up to date", "Scholarship holder", "Mother's occupation", "Father's occupation", "Mother's qualification", "Father's qualification"]
cols2 = [c for c in MOMENTS["τέλος 1ου εξαμήνου"] if c not in SOCIO]
m2, X2, risk2 = fit(cols2)
T2 = D.loc[te].assign(risk=risk2)
print("AUC με όλα:", round(roc_auc_score(T.y, T.risk), 3), "| AUC χωρίς τα κοινωνικοοικονομικά:", round(roc_auc_score(T2.y, T2.risk), 3))
for name, TT in (("με όλα", T), ("χωρίς", T2)):
    late = TT[TT["Tuition fees up to date"] == 0]; r = perf(late, late.risk >= 0.4)
    print(name, "· δίδακτρα σε καθυστέρηση: επισημαίνονται", r["επισημαίνονται"], "από", r["φοιτητές"], "· ψευδώς θετικοί", r["ψευδώς θετικοί %"], "% · ψευδώς αρνητικοί", r["ψευδώς αρνητικοί %"], "%")

> **Σχολιασμός.** Η AUC πέφτει από 0.890 σε 0.875. Για όσους καθυστερούν τα δίδακτρα, οι ψευδώς θετικοί πέφτουν από 56% σε 6% και οι ψευδώς αρνητικοί ανεβαίνουν από 3% σε 25%. Η ελαχιστοποίηση έχει τίμημα σε ακρίβεια και αλλάζει το ποιος αδικείται· δεν εξαφανίζει την αδικία. Αυτό είναι το σημείο της άσκησης.